# PaperMind - ByT5 OCR-correction fine-tune (v2)

Trains a model that repairs TrOCR's handwriting-recognition errors line by line. **Runtime -> Change runtime type -> A100 GPU.**

Run the cells top to bottom. **First run with `SMOKE = True` (about 5 minutes)** to confirm everything works, then set `SMOKE = False` for the real run.

**Before you start:** zip the folder `backend/training/correction` of the project and upload it to Google Drive as
`MyDrive/papermind/papermind_correction.zip` (or run `python backend/training/make_zip.py`).

Why this replaces the old notebook: nothing is version-pinned (the old cell 1 pinned `tokenizers==0.19.1`, which has no
prebuilt wheel on current Colab and hangs while compiling Rust), the splits are by sentence so test scores are honest,
and it checks that the model does **not** "correct" a student's factual mistakes.

In [ ]:
# Cell 1 - install. NO version pins: use the transformers/torch/datasets that Colab already ships.
!pip -q install rapidfuzz
import torch, transformers, datasets
print('torch', torch.__version__, '| transformers', transformers.__version__, '| datasets', datasets.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - switch the runtime to A100')

In [ ]:
# Cell 2 - settings
SMOKE = True            # True: ~5 min sanity run.  False: the real run.

if SMOKE:
    N_LINES, WIKI_N, MODEL, EPOCHS, BS = 3000, 1500, 'google/byt5-small', 1, 32
else:
    N_LINES, WIKI_N, MODEL, EPOCHS, BS = 60000, 20000, 'google/byt5-base', 4, 64

TROCR_FRAC = 0.35       # share of training lines noised by the REAL TrOCR model (rest: fast simulated noise)
USE_IAM = False         # True: also train on REAL handwriting (IAM) - needs the Kaggle token, see the next cell
IAM_LINES = 8000        # how many IAM lines to add
DRIVE = '/content/drive/MyDrive/papermind'
RUN = 'smoke' if SMOKE else 'full'
print(f'{RUN}: {N_LINES:,} lines | {MODEL} | {EPOCHS} epoch(s) | batch {BS}')

In [ ]:
# Cell 3 - mount Drive, unpack the code
from google.colab import drive
drive.mount('/content/drive')
import os
zip_path = f'{DRIVE}/papermind_correction.zip'
assert os.path.exists(zip_path), f'Upload papermind_correction.zip to {DRIVE}/ first'
!rm -rf /content/pm && mkdir -p /content/pm && unzip -q -o "{zip_path}" -d /content/pm
!ls /content/pm

In [ ]:
# Cell 4 - handwriting-style fonts (open licence) so TrOCR sees realistic, varied writing
!mkdir -p /content/pm/fonts
base = 'https://github.com/google/fonts/raw/main/ofl'
for name, path in [('IndieFlower', 'indieflower/IndieFlower-Regular.ttf'), ('PatrickHand', 'patrickhand/PatrickHand-Regular.ttf'),
                   ('Kalam', 'kalam/Kalam-Regular.ttf'), ('ShadowsIntoLight', 'shadowsintolight/ShadowsIntoLight.ttf')]:
    !curl -sL "{base}/{path}" -o /content/pm/fonts/{name}.ttf
!ls -la /content/pm/fonts

In [ ]:
# Cell 4b - OPTIONAL: real handwriting from Kaggle (IAM). Skip this cell when USE_IAM = False.
# One-time setup: Kaggle -> Settings -> API -> "Create New Token" (downloads kaggle.json). Then in Colab, left sidebar ->
# key icon (Secrets) -> add  KAGGLE_USERNAME  and  KAGGLE_KEY  (values from kaggle.json) and switch "Notebook access" on.
# IAM terms: non-commercial research use; cite Marti & Bunke (2002).
IAM_ARG = ''
if USE_IAM:
    import os
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    import kagglehub
    IAM_DIR = kagglehub.dataset_download('maxicode/iam-handwriting')   # full IAM release; can be several GB
    print('IAM at', IAM_DIR)
    !ls "{IAM_DIR}"
    IAM_ARG = f'--iam-dir {IAM_DIR} --iam-lines {IAM_LINES}'

In [ ]:
# Cell 5 - build the dataset (real TrOCR + simulated noise, split by sentence). Add your own syllabus/textbook PDFs
# or student answers with  --files /content/drive/MyDrive/papermind/syllabus.pdf  to make it more in-domain.
!cd /content/pm && python build_dataset.py --out /content/data --n-lines {N_LINES} --wiki-n {WIKI_N} \
    --font-dir /content/pm/fonts --trocr-frac {TROCR_FRAC} --trocr-batch 128 {IAM_ARG}
!mkdir -p "{DRIVE}/data_{RUN}" && cp /content/data/* "{DRIVE}/data_{RUN}/"

In [ ]:
# Cell 6 - train (SMOKE: a couple of minutes.  FULL byt5-base on A100: roughly 30-60 minutes)
!cd /content/pm && python train.py --data /content/data --out /content/ckpt_{RUN} --model {MODEL} --epochs {EPOCHS} --bs {BS}

In [ ]:
# Cell 7 - read the report. What to look at:
#   overall.cer_after  vs  overall.cer_before      -> must go DOWN
#   over_correction.rate                           -> must be close to 0 (lines that were already right must not change)
#   preservation.rate                              -> must be close to 0 (model must not rewrite a student's wrong fact)
import json
r = json.load(open(f'/content/ckpt_{RUN}/test_report.json'))
for k, v in r.items():
    print(k, v)

In [ ]:
# Cell 8 - try it on the errors seen in the real PaperMind demo
import sys; sys.path.insert(0, '/content/pm')
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from evaluate_correction import correct_texts
tok = AutoTokenizer.from_pretrained(f'/content/ckpt_{RUN}')
model = AutoModelForSeq2SeqLM.from_pretrained(f'/content/ckpt_{RUN}').to('cuda')
tests = ['19 ( 0b ) .', '2 ) False', '3 ) Liskov . principle says objects of a', 'program . Square extending . i Rectangle violates',
         'A stack uses First In First Out ordering.', 'Method overriding is resolved at compile time.']
for a, b in zip(tests, correct_texts(model, tok, tests, beams=4)):
    print(repr(a), '->', repr(b))

In [ ]:
# Cell 9 - save to Drive (only meaningful after the FULL run) and zip for download
!mkdir -p "{DRIVE}/byt5-correction-{RUN}" && cp -r /content/ckpt_{RUN}/* "{DRIVE}/byt5-correction-{RUN}/"
!cd /content && rm -f /content/byt5-correction-{RUN}.zip && zip -qr /content/byt5-correction-{RUN}.zip ckpt_{RUN} -x "ckpt_{RUN}/checkpoint-*"
from google.colab import files
files.download(f'/content/byt5-correction-{RUN}.zip')

## Use it in PaperMind
1. Unzip so that `config.json`, `model.safetensors`, `tokenizer_config.json` ... sit in `C:\PaperMind\backend\models\byt5-correction\`.
2. In `backend/.env` add `CORRECTION_MODEL_PATH=./models/byt5-correction` and restart the backend.
3. The backend keeps the original OCR text for any line the model rewrites too aggressively (`apply_guard`), so a bad
   correction can never replace what the student wrote.